# Real-Time Voice Activity Detection (VAD) - Demo

This notebook demonstrates:
- Voice Activity Detection on audio files
- Energy-based detection
- Chunk processing for real-time simulation
- Visualizing speech regions

In [ ]:
# Install required packages
!pip install librosa soundfile numpy matplotlib

In [ ]:
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import time

print("Libraries loaded successfully")

## 1. Generate Test Audio

In [ ]:
def generate_test_audio(filename="test_vad.wav", sr=16000, duration=5):
    """Generate audio with speech and silence segments"""
    t = np.linspace(0, duration, int(sr * duration))
    
    audio = np.zeros(int(sr * duration))
    
    # Speech segments (simulated with sine waves)
    # 0-1s: silence
    # 1-2s: speech
    for i in range(1, 2):
        start_sample = i * sr
        end_sample = (i + 1) * sr
        speech = 0.5 * np.sin(2 * np.pi * 300 * t[start_sample:end_sample])
        speech += 0.3 * np.sin(2 * np.pi * 500 * t[start_sample:end_sample])
        audio[start_sample:end_sample] = speech
    
    # 2-3s: silence
    
    # 3-4s: speech
    for i in range(3, 4):
        start_sample = i * sr
        end_sample = (i + 1) * sr
        speech = 0.4 * np.sin(2 * np.pi * 250 * t[start_sample:end_sample])
        audio[start_sample:end_sample] = speech
    
    # 4-5s: silence
    
    sf.write(filename, audio, sr)
    print(f"Created: {filename}")
    return filename

test_file = generate_test_audio()

## 2. Energy-Based VAD

In [ ]:
class EnergyVAD:
    """Voice Activity Detection using energy"""
    
    def __init__(self, energy_threshold=0.01):
        self.energy_threshold = energy_threshold
    
    def detect(self, audio_chunk):
        """Detect if chunk contains speech"""
        energy = np.sum(audio_chunk ** 2) / len(audio_chunk)
        is_speech = energy > self.energy_threshold
        return is_speech, energy
    
    def detect_file(self, audio, sr, chunk_duration=0.5):
        """Process entire file"""
        chunk_samples = int(chunk_duration * sr)
        results = []
        
        for i in range(0, len(audio), chunk_samples):
            chunk = audio[i:i + chunk_samples]
            if len(chunk) < chunk_samples:
                break
            
            is_speech, energy = self.detect(chunk)
            results.append({
                'start_time': i / sr,
                'end_time': (i + chunk_samples) / sr,
                'is_speech': is_speech,
                'energy': energy
            })
        
        return results

# Load audio
audio, sr = librosa.load(test_file, sr=16000)
print(f"Audio loaded: {len(audio)/sr:.2f}s at {sr}Hz")

# Run VAD
vad = EnergyVAD(energy_threshold=0.02)
results = vad.detect_file(audio, sr, chunk_duration=0.5)

# Print results
print("\n=== VAD Results ===")
for r in results:
    status = "🔊 SPEECH" if r['is_speech'] else "  -- SILENCE"
    print(f"{r['start_time']:.1f}s - {r['end_time']:.1f}s: {status} (energy: {r['energy']:.4f})")

## 3. Visualize VAD Results

In [ ]:
# Plot waveform with VAD results
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

time_axis = np.arange(len(audio)) / sr

# Plot audio
plt.subplot(2, 1, 1)
plt.plot(time_axis, audio, 'b-', alpha=0.7)
plt.title('Audio Waveform')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)

# Plot VAD
plt.subplot(2, 1, 2)
colors = []
for r in results:
    colors.append('green' if r['is_speech'] else 'red')

for i, r in enumerate(results):
    plt.axvspan(r['start_time'], r['end_time'], 
                alpha=0.3, color='green' if r['is_speech'] else 'red')

plt.title('Voice Activity Detection')
plt.xlabel('Time (s)')
plt.grid(True, alpha=0.3)
plt.xlim(0, duration)
plt.tight_layout()
plt.show()

## 4. Performance Test

In [ ]:
# Test real-time performance
def test_realtime_performance(audio, sr, num_runs=100):
    """Test VAD speed"""
    chunk_size = 512  # ~32ms
    
    start = time.time()
    for _ in range(num_runs):
        chunk = audio[:chunk_size]
        _ = vad.detect(chunk)
    elapsed = time.time() - start
    
    avg_time = (elapsed / num_runs) * 1000  # ms
    
    print(f"=== Real-Time Performance ===")
    print(f"Chunk size: {chunk_size/sr*1000:.0f}ms")
    print(f"Avg inference: {avg_time:.3f}ms")
    print(f"Can run real-time: {'Yes' if avg_time < chunk_size/sr*1000 else 'No'}")

duration = len(audio) / sr
test_realtime_performance(audio, sr)

## 🎯 Summary

This notebook shows:
- Energy-based Voice Activity Detection
- Chunk processing for real-time feel
- Visualization of speech regions
- Performance measurement

This VAD can be used for:
- Edge devices (earbuds, IoT)
- Real-time streaming
- Pre-processing before enhancement